In [1]:
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score

In [2]:
DATA_PATH = "DiseaseAndSymptoms.csv"   # update path as needed
OUT_DIR = "C:/Users/Mohan Raj P/OneDrive/Desktop/Dataset"

In [3]:
cd = "C:\\Users\\Mohan Raj P\\OneDrive\\Desktop\\Dataset\\DiseaseAndSymptoms.csv"

In [4]:
df = pd.read_csv(DATA_PATH)
symptom_cols = [c for c in df.columns if c.startswith("Symptom")]

In [5]:
for c in symptom_cols:
    df[c] = df[c].apply(lambda x: x.strip() if isinstance(x, str) else x)
df["Disease"] = df["Disease"].str.strip()

In [13]:
FORM_SYMPTOMS = [

    "mild_fever", "high_fever", "cough", "runny_nose", "breathlessness",
    "fatigue", "headache", "muscle_pain", "throat_irritation", "nausea",
    "vomiting", "diarrhoea", "loss_of_appetite", "chest_pain", "chills",
    "dizziness", "joint_pain", "skin_rash", "polyuria",
    "blurred_and_distorted_vision",

    "abdominal_pain", "yellowish_skin", "yellowing_of_eyes", "malaise",
    "itching", "sweating", "dark_urine", "irritability",
    "excessive_hunger", "weight_loss", "lethargy", "phlegm",
    "swelled_lymph_nodes", "loss_of_balance", "abnormal_menstruation",
    "muscle_weakness", "depression", "fast_heart_rate",
    "red_spots_over_body", "back_pain",
]
 
df = pd.read_csv(DATA_PATH)
symptom_cols = [c for c in df.columns if c.startswith("Symptom")]
for c in symptom_cols:
    df[c] = df[c].apply(lambda x: x.strip() if isinstance(x, str) else x)
df["Disease"] = df["Disease"].str.strip()
 
# Restrict to only what the (expanded) form will ever collect
symptom_lists = df[symptom_cols].apply(
    lambda row: sorted({v for v in row if pd.notna(v) and v in FORM_SYMPTOMS}),
    axis=1,
)
 
mlb = MultiLabelBinarizer(classes=FORM_SYMPTOMS)
X = mlb.fit_transform(symptom_lists)
le = LabelEncoder()
y = le.fit_transform(df["Disease"])
 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
 
rf = RandomForestClassifier(n_estimators=100, random_state=42)
nb = GaussianNB()
svm = SVC(kernel="rbf", probability=True, random_state=42)
 
for name, model in [("RandomForest", rf), ("NaiveBayes", nb), ("SVM", svm)]:
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    cv = cross_val_score(model, X, y, cv=5).mean()
    print(f"{name}: test_acc={acc:.4f}  5-fold_cv_acc={cv:.4f}")
 
joblib.dump(mlb, f"{OUT_DIR}/expanded_form_encoder.joblib")
joblib.dump(le, f"{OUT_DIR}/expanded_form_label_encoder.joblib")
joblib.dump(rf, f"{OUT_DIR}/expanded_form_rf.joblib")
joblib.dump(nb, f"{OUT_DIR}/expanded_form_nb.joblib")
joblib.dump(svm, f"{OUT_DIR}/expanded_form_svm.joblib")
print(f"\nSaved expanded-form models ({len(FORM_SYMPTOMS)} symptoms) to {OUT_DIR}/")
 

RandomForest: test_acc=0.9278  5-fold_cv_acc=0.9305
NaiveBayes: test_acc=0.9096  5-fold_cv_acc=0.9118
SVM: test_acc=0.9278  5-fold_cv_acc=0.9305

Saved expanded-form models (40 symptoms) to C:/Users/Mohan Raj P/OneDrive/Desktop/Dataset/


In [6]:
symptom_lists = df[symptom_cols].apply(lambda row: sorted({v for v in row if pd.notna(v)}), axis=1)
 
mlb = MultiLabelBinarizer()
X = mlb.fit_transform(symptom_lists)
print(f"Feature count: {X.shape[1]} (should equal the number of unique "
      f"symptoms in the dataset, NOT 394)")
 
le = LabelEncoder()
y = le.fit_transform(df["Disease"])

Feature count: 131 (should equal the number of unique symptoms in the dataset, NOT 394)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
 

In [8]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
nb = GaussianNB()
svm = SVC(kernel="rbf", probability=True, random_state=42)
 
for name, model in [("RandomForest", rf), ("NaiveBayes", nb), ("SVM", svm)]:
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    cv = cross_val_score(model, X, y, cv=5).mean()
    print(f"\n{name}: test_acc={acc:.4f}  5-fold_cv_acc={cv:.4f}")
 


RandomForest: test_acc=1.0000  5-fold_cv_acc=1.0000

NaiveBayes: test_acc=1.0000  5-fold_cv_acc=1.0000

SVM: test_acc=1.0000  5-fold_cv_acc=1.0000


In [9]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
nb = GaussianNB()
svm = SVC(kernel="rbf", probability=True, random_state=42)
 
for name, model in [("RandomForest", rf), ("NaiveBayes", nb), ("SVM", svm)]:
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    cv = cross_val_score(model, X, y, cv=5).mean()
    print(f"\n{name}: test_acc={acc:.4f}  5-fold_cv_acc={cv:.4f}")
 


RandomForest: test_acc=1.0000  5-fold_cv_acc=1.0000

NaiveBayes: test_acc=1.0000  5-fold_cv_acc=1.0000

SVM: test_acc=1.0000  5-fold_cv_acc=1.0000


In [10]:
def predict_from_symptoms(symptom_list, model):
    """symptom_list: python list of raw symptom tokens, any order, any length."""
    vec = mlb.transform([symptom_list])
    pred = model.predict(vec)
    return le.inverse_transform(pred)[0]
 
sample_row = df.iloc[0]
sample_symptoms = [v for v in sample_row[symptom_cols] if pd.notna(v)]
shuffled = sample_symptoms[::-1]  # reverse order, still same symptoms
print("\nOriginal order prediction:", predict_from_symptoms(sample_symptoms, rf))
print("Reversed order prediction :", predict_from_symptoms(shuffled, rf))
print("(These should match. In the buggy 394-feature model they usually won't.)")


Original order prediction: Fungal infection
Reversed order prediction : Fungal infection
(These should match. In the buggy 394-feature model they usually won't.)


In [11]:
mlb = joblib.load("form_matched_encoder.joblib")
label_encoder = joblib.load("form_matched_label_encoder.joblib")
rf = joblib.load("form_matched_rf.joblib")
nb = joblib.load("form_matched_nb.joblib")
svm = joblib.load("form_matched_svm.joblib")

c:\Users\Mohan Raj P\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MultiLabelBinarizer from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Mohan Raj P\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Mohan Raj P\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.8.0 when using version 1.7.2

In [12]:
with open(f"{OUT_DIR}/symptom_vocabulary.json", "w") as f:
    json.dump(sorted(mlb.classes_.tolist()), f, indent=2)
 
print(f"\nSaved encoder + models + {len(mlb.classes_)}-symptom vocabulary to {OUT_DIR}/")


Saved encoder + models + 20-symptom vocabulary to C:/Users/Mohan Raj P/OneDrive/Desktop/Dataset/


In [14]:
FORM_SYMPTOMS = [
    "mild_fever", "high_fever", "cough", "runny_nose", "breathlessness",
    "fatigue", "headache", "muscle_pain", "throat_irritation", "nausea",
    "vomiting", "diarrhoea", "loss_of_appetite", "chest_pain", "chills",
    "dizziness", "joint_pain", "skin_rash", "polyuria",
    "blurred_and_distorted_vision",
    
    "abdominal_pain", "yellowish_skin", "yellowing_of_eyes", "malaise",
    "itching", "sweating", "dark_urine", "irritability",
    "excessive_hunger", "weight_loss", "lethargy", "phlegm",
    "swelled_lymph_nodes", "loss_of_balance", "abnormal_menstruation",
    "muscle_weakness", "depression", "fast_heart_rate",
    "red_spots_over_body", "back_pain",
]
 
df = pd.read_csv(DATA_PATH)
symptom_cols = [c for c in df.columns if c.startswith("Symptom")]
for c in symptom_cols:
    df[c] = df[c].apply(lambda x: x.strip() if isinstance(x, str) else x)
df["Disease"] = df["Disease"].str.strip()
 
symptom_lists = df[symptom_cols].apply(
    lambda row: sorted({v for v in row if pd.notna(v) and v in FORM_SYMPTOMS}),
    axis=1,
)
 
mlb = MultiLabelBinarizer(classes=FORM_SYMPTOMS)
X = mlb.fit_transform(symptom_lists)
le = LabelEncoder()
y = le.fit_transform(df["Disease"])
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
 
rf = RandomForestClassifier(n_estimators=100, random_state=42)
nb = GaussianNB()
svm = SVC(kernel="rbf", probability=True, random_state=42)
 
for name, model in [("RandomForest", rf), ("NaiveBayes", nb), ("SVM", svm)]:
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    cv = cross_val_score(model, X, y, cv=5).mean()
    print(f"{name}: test_acc={acc:.4f}  5-fold_cv_acc={cv:.4f}")
 
joblib.dump(mlb, f"{OUT_DIR}/expanded_form_encoder.joblib")
joblib.dump(le, f"{OUT_DIR}/expanded_form_label_encoder.joblib")
joblib.dump(rf, f"{OUT_DIR}/expanded_form_rf.joblib")
joblib.dump(nb, f"{OUT_DIR}/expanded_form_nb.joblib")
joblib.dump(svm, f"{OUT_DIR}/expanded_form_svm.joblib")
print(f"\nSaved expanded-form models ({len(FORM_SYMPTOMS)} symptoms) to {OUT_DIR}/")
 

RandomForest: test_acc=0.9278  5-fold_cv_acc=0.9305
NaiveBayes: test_acc=0.9096  5-fold_cv_acc=0.9118
SVM: test_acc=0.9278  5-fold_cv_acc=0.9305

Saved expanded-form models (40 symptoms) to C:/Users/Mohan Raj P/OneDrive/Desktop/Dataset/


In [17]:
CONFIDENCE_BANDS = [
    (0.70, "HIGH"),
    (0.50, "MEDIUM"),
    (0.00, "LOW"),
]
AMBIGUOUS_MARGIN_THRESHOLD = 0.25
 

CHECKBOX_TO_SYMPTOMS = {
    "Mild Fever": ["mild_fever"],
    "High Fever": ["high_fever"],
    "Cough": ["cough"],
    "Runny Nose": ["runny_nose"],
    "Breathlessness": ["breathlessness"],
    "Fatigue": ["fatigue"],
    "Headache": ["headache"],
    "Muscle Pain": ["muscle_pain"],
    "Throat Irritation": ["throat_irritation"],
    "Nausea": ["nausea"],
    "Vomiting": ["vomiting"],
    "Diarrhoea": ["diarrhoea"],
    "Loss of Appetite": ["loss_of_appetite"],
    "Chest Pain": ["chest_pain"],
    "Chills": ["chills"],
    "Dizziness": ["dizziness"],
    "Joint Pain": ["joint_pain"],
    "Skin Rash": ["skin_rash"],
    "Polyuria": ["polyuria"],
    "Blurred and Distorted Vision": ["blurred_and_distorted_vision"],
    "Abdominal Pain": ["abdominal_pain"],
    "Yellowish Skin": ["yellowish_skin"],
    "Yellowing of Eyes": ["yellowing_of_eyes"],
    "Malaise": ["malaise"],
    "Itching": ["itching"],
    "Sweating": ["sweating"],
    "Dark Urine": ["dark_urine"],
    "Irritability": ["irritability"],
    "Excessive Hunger": ["excessive_hunger"],
    "Weight Loss": ["weight_loss"],
    "Lethargy": ["lethargy"],
    "Phlegm": ["phlegm"],
    "Swelled Lymph Nodes": ["swelled_lymph_nodes"],
    "Loss of Balance": ["loss_of_balance"],
    "Abnormal Menstruation": ["abnormal_menstruation"],
    "Muscle Weakness": ["muscle_weakness"],
    "Depression": ["depression"],
    "Fast Heart Rate": ["fast_heart_rate"],
    "Red Spots Over Body": ["red_spots_over_body"],
    "Back Pain": ["back_pain"],
}
 

 

CONFUSABLE_GROUPS = [
    {"Arthritis", "Osteoarthristis"},
    {"Hepatitis B", "Hepatitis C", "Hepatitis D", "Hepatitis E", "hepatitis A",
     "Chronic cholestasis", "Peptic ulcer diseae"},
    {"Hyperthyroidism", "Hypothyroidism", "Varicose veins"},
]
 

EMERGENCY_DISEASES = {"Heart attack", "Paralysis (brain hemorrhage)"}
 
 
def checkboxes_to_symptom_vector(checked_boxes):
    tokens = set()
    dropped = []
    for box in checked_boxes:
        mapped = CHECKBOX_TO_SYMPTOMS.get(box)
        if not mapped:
            dropped.append(box)
            continue
        tokens.update(mapped)
    vec = mlb.transform([sorted(tokens)])
    return vec, sorted(tokens), dropped
 
 
def confidence_band(p):
    for threshold, label in CONFIDENCE_BANDS:
        if p >= threshold:
            return label
    return "LOW"
 
 
def predict_guarded(checked_boxes, top_n=3, min_symptoms=2):
    vec, matched_tokens, dropped = checkboxes_to_symptom_vector(checked_boxes)
 
    proba_rf = rf.predict_proba(vec)[0]
    proba_nb = nb.predict_proba(vec)[0]
    proba_svm = svm.predict_proba(vec)[0]
    avg_proba = (proba_rf + proba_nb + proba_svm) / 3
 
    order = np.argsort(avg_proba)[::-1]
    top_idx = order[:top_n]
    ranked = [
        {"disease": label_encoder.inverse_transform([i])[0],
         "confidence": round(float(avg_proba[i]), 4)}
        for i in top_idx
    ]
    top1_disease = ranked[0]["disease"]
    top1_conf = ranked[0]["confidence"]
    margin = float(avg_proba[order[0]] - avg_proba[order[1]])
 
    # per-model agreement
    top1_rf = label_encoder.inverse_transform([proba_rf.argmax()])[0]
    top1_nb = label_encoder.inverse_transform([proba_nb.argmax()])[0]
    top1_svm = label_encoder.inverse_transform([proba_svm.argmax()])[0]
    models_agree = len({top1_rf, top1_nb, top1_svm}) == 1
 
    flags = []
    band = confidence_band(top1_conf)
    too_few = len(matched_tokens) < min_symptoms
    if too_few:
        
        band = "LOW"
        flags.append("TOO_FEW_SYMPTOMS")
    if band != "HIGH":
        flags.append(f"{band}_CONFIDENCE")
    if margin < AMBIGUOUS_MARGIN_THRESHOLD:
        flags.append("AMBIGUOUS_TOP_CANDIDATES")
    if not models_agree:
        flags.append("MODEL_DISAGREEMENT")
 
    confusable_note = None
    for group in CONFUSABLE_GROUPS:
        if top1_disease in group:
            others_in_topn = [p["disease"] for p in ranked[1:] if p["disease"] in group]
            confusable_note = (
                f"'{top1_disease}' cannot be reliably separated from "
                f"{sorted(group - {top1_disease})} using only checkbox symptoms."
            )
            if not others_in_topn:
               
                pass
            break
 
    emergency_hits = [p["disease"] for p in ranked if p["disease"] in EMERGENCY_DISEASES]
 
    # human-readable recommendation
    if emergency_hits:
        recommendation = (
            f"URGENT: {', '.join(emergency_hits)} appears among the top possibilities. "
            f"Recommend immediate in-person clinical evaluation regardless of model confidence."
        )
    elif "TOO_FEW_SYMPTOMS" in flags:
        recommendation = "Not enough symptoms selected for a meaningful prediction. Ask the patient for more."
    elif band == "HIGH" and models_agree and margin >= AMBIGUOUS_MARGIN_THRESHOLD:
        recommendation = "High-confidence prediction. Still confirm with a clinician before treatment."
    else:
        recommendation = (
            "Low-confidence / ambiguous result. Treat this as a shortlist, not a diagnosis -- "
            "refer to a clinician for confirmation."
        )
 
    return {
        "predictions": ranked,
        "confidence_band": band,
        "top1_vs_top2_margin": round(margin, 4),
        "model_agreement": {"rf": top1_rf, "nb": top1_nb, "svm": top1_svm, "all_agree": models_agree},
        "flags": flags,
        "confusable_with_note": confusable_note,
        "emergency_alert": bool(emergency_hits),
        "recommendation": recommendation,
        "matched_symptoms": matched_tokens,
        "ignored_checkboxes": dropped,
        "disclaimer": "This is a screening aid, not a medical diagnosis.",
    }
 
 
if __name__ == "__main__":
    print("--- High-confidence, unambiguous case ---")
    r1 = predict_guarded(["Dizziness", "Nausea", "Vomiting", "Headache"])
    for k, v in r1.items():
        print(f"  {k}: {v}")
 
    print("\n--- Known-confusable case (Arthritis vs Osteoarthritis) ---")
    r2 = predict_guarded(["Joint Pain", "Muscle Pain"])
    for k, v in r2.items():
        print(f"  {k}: {v}")
 
    print("\n--- Too few symptoms ---")
    r3 = predict_guarded(["Headache"])
    for k, v in r3.items():
        print(f"  {k}: {v}")
 

--- High-confidence, unambiguous case ---
  predictions: [{'disease': '(vertigo) Paroymsal  Positional Vertigo', 'confidence': 0.6844}, {'disease': 'Hypertension', 'confidence': 0.1101}, {'disease': 'Paralysis (brain hemorrhage)', 'confidence': 0.0215}]
  confidence_band: MEDIUM
  top1_vs_top2_margin: 0.5742
  model_agreement: {'rf': '(vertigo) Paroymsal  Positional Vertigo', 'nb': '(vertigo) Paroymsal  Positional Vertigo', 'svm': '(vertigo) Paroymsal  Positional Vertigo', 'all_agree': True}
  flags: ['MEDIUM_CONFIDENCE']
  confusable_with_note: None
  emergency_alert: True
  recommendation: URGENT: Paralysis (brain hemorrhage) appears among the top possibilities. Recommend immediate in-person clinical evaluation regardless of model confidence.
  matched_symptoms: ['dizziness', 'headache', 'nausea', 'vomiting']
  ignored_checkboxes: []
  disclaimer: This is a screening aid, not a medical diagnosis.

--- Known-confusable case (Arthritis vs Osteoarthritis) ---
  predictions: [{'disease':